<a href="https://colab.research.google.com/github/100522371/P1_AA_Grupo17/blob/main/predicciones.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Montserrat Martis Contreras - 100522371

Laith Basem Elbeshti - 100477240

In [5]:
import pandas as pd
import joblib

# Para que funcione, debe estar ya en los Archivos del notebook
model = joblib.load('modelo_final.joblib')

In [6]:
df = pd.read_pickle('bank_competition.pkl')
df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome
5553,43,management,married,tertiary,no,78,yes,no,cellular,21,nov,36,1,109,1,other
915,34,housemaid,married,secondary,no,0,yes,no,unknown,30,oct,154,1,-1,0,unknown
7652,54,technician,married,secondary,no,3323,yes,yes,cellular,8,apr,59,3,-1,0,unknown
5065,43,blue-collar,single,primary,no,-399,no,yes,cellular,28,jul,662,3,-1,0,unknown
3338,35,blue-collar,married,secondary,no,262,no,no,cellular,15,mar,427,1,181,3,success


Tenemos que realizar el mismo preprocesado de *pdays* con los datos a predecir

In [7]:
import numpy as np

# 1. Definimos los cortes (bins)
cortes = [-np.inf, -1, 200, 400, np.inf]

# 2. Definimos las etiquetas
etiquetas = [
    'no_contact',
    'recent',
    'intermediate',
    'old'
]

# 3. Sobrescribimos la columna original directamente
df['pdays'] = pd.cut(df['pdays'], bins=cortes, labels=etiquetas, right=True)

# Comprobamos el resultado
print(df['pdays'].value_counts())

pdays
no_contact      121
recent           21
intermediate     19
old               1
Name: count, dtype: int64


In [8]:
df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome
5553,43,management,married,tertiary,no,78,yes,no,cellular,21,nov,36,1,recent,1,other
915,34,housemaid,married,secondary,no,0,yes,no,unknown,30,oct,154,1,no_contact,0,unknown
7652,54,technician,married,secondary,no,3323,yes,yes,cellular,8,apr,59,3,no_contact,0,unknown
5065,43,blue-collar,single,primary,no,-399,no,yes,cellular,28,jul,662,3,no_contact,0,unknown
3338,35,blue-collar,married,secondary,no,262,no,no,cellular,15,mar,427,1,recent,3,success


In [9]:
predicciones = model.predict(df)

df_entrega = pd.DataFrame({
    'ID': df.index,
    'predicciones': predicciones
})

df_entrega.to_csv('predicciones.csv', index=False)

## Despliegue Streamlit

In [10]:
# Crear un DataFrame con las dos instancias que probamos en Streamlit
datos_nuevos = {
    'age': [28, 42],
    'job': ['services', 'management'],
    'marital': ['single', 'married'],
    'education': ['tertiary', 'tertiary'],
    'default': ['no', 'no'],
    'balance': [1200, 1800],
    'housing': ['no', 'yes'],
    'loan': ['yes', 'no'],
    'contact': ['cellular', 'telephone'],
    'day': [23, 12],
    'month': ['jan', 'mar'],
    'duration': [119, 174],
    'campaign': [1, 3],
    'pdays': [-1, 100],
    'previous': [0, 1],
    'poutcome': ['other', 'unknown']
}

df_instancias = pd.DataFrame(datos_nuevos)
display(df_instancias)

# Sobrescribimos pdays
df_instancias['pdays'] = pd.cut(df_instancias['pdays'], bins=cortes, labels=etiquetas, right=True)
display(df_instancias)

# Realizar las predicciones con el Pipeline
# El pipeline se encarga automáticamente de escalar y categorizar (OneHot, pdays, etc.)
predicciones = model.predict(df_instancias)
probabilidades = model.predict_proba(df_instancias)

# Mostramos los resultados con el mismo formato que la App web
for i in range(len(df_instancias)):
    prob_no = probabilidades[i][0] * 100
    prob_si = probabilidades[i][1] * 100
    pred_texto = "SÍ contratará" if predicciones[i] == 'yes' else "NO contratará"

    print(f"INSTANCIA {i+1}:")
    print(f"El modelo predice: {pred_texto} el depósito.")
    print(f"Probabilidad de No contrata (no): {prob_no:.2f}%")
    print(f"Probabilidad de Sí contrata (yes): {prob_si:.2f}%")

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome
0,28,services,single,tertiary,no,1200,no,yes,cellular,23,jan,119,1,-1,0,other
1,42,management,married,tertiary,no,1800,yes,no,telephone,12,mar,174,3,100,1,unknown


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome
0,28,services,single,tertiary,no,1200,no,yes,cellular,23,jan,119,1,no_contact,0,other
1,42,management,married,tertiary,no,1800,yes,no,telephone,12,mar,174,3,recent,1,unknown


INSTANCIA 1:
El modelo predice: NO contratará el depósito.
Probabilidad de No contrata (no): 84.87%
Probabilidad de Sí contrata (yes): 15.13%
INSTANCIA 2:
El modelo predice: SÍ contratará el depósito.
Probabilidad de No contrata (no): 20.12%
Probabilidad de Sí contrata (yes): 79.88%
